# Dataset class and helper functions

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional, Sequence, Literal

import math
import random

import numpy as np
import pandas as pd

In [2]:
@dataclass
class BlockSample:
    subject_id: str
    block_id: int
    start_idx: int
    end_idx: int

In [3]:
class SleepDataset:
    """
    Each input parquet file is assumed to contain one subject's time-ordered windowed data:
      - time column
      - feature columns
      - label columns

    The dataset indexes contiguous blocks within each subject and returns:
      X_block: (n_samples, n_features)
      y_block: (n_samples,)
      meta: dict with subject/block info
    """
    def __init__(
            self,
            signals_dir: str | Path,
            metadata_csv: str | Path,
            feature_cols: Sequence[str],
            label_cols: Sequence[str],
            subject_col_meta: str = "sid",
            signals_file_pattern: str = "{subject_id}.parquet",
            block_duration_sec: float = 30.0*60.0,  # 30 minutes
            epoch_sec: Optional[float] = 5.0, # 5s epochs
            horizon: int = 0,
            allowed_subjects: Optional[Sequence[str]] = None,
            drop_boundary: int = 25//5, # assume EXTENDED_EPOCH=25s, EPOCH=5s, horizon=0
            tail_policy: str = "drop",
            preload: bool = False,
    ) -> None:
        """
        Parameters
        ----------
        signals_dir:
            Folder containing one parquet file per subject.
        metadata_csv:
            CSV with subject-level metadata.
        feature_cols:
            Names of feature columns to use from parquet.
        label_cols:
            Names of label columns in parquet (sleep stage, apnea events, etc.).
        subject_col_meta:
            Subject id column in metadata csv.
        signals_file_pattern:
            Pattern for parquet filenames, e.g. "{subject_id}.parquet".
        block_duration_sec:
            Duration of each block in seconds. 
        epoch_sec:
            Duration represented by each row/epoch.
        horizon:
            Future prediction shift. y[t] becomes label at t + horizon.
        allowed_subjects:
            List of allowed sid.
        drop_boundary:
            Number of windows to drop at both / beginning sides of each block boundary.
            Minimal version keeps this simple by shrinking each block internally.
        tail_policy:
            Decide what to do with the remainder len(df)%block_size.
            "merge": merge remainder epochs into last block
            "pad" : add remainder with padding to new block such that all blocks are same length
            "drop" : drop remainder epochs
        preload:
            If True, load all subject data at init. Otherwise lazy-load.
        """
        self.signals_dir = Path(signals_dir)
        self.metadata = pd.read_csv(metadata_csv)
        self.feature_cols = list(feature_cols)
        self.label_cols = list(label_cols)
        self.subject_col_meta = subject_col_meta
        self.signals_file_pattern = signals_file_pattern
        self.preload = preload
        
        if block_duration_sec <= epoch_sec:
            raise ValueError(f"block_duration_sec must be >= epoch_sec {epoch_sec}s")
        self.block_size = int(block_duration_sec//epoch_sec)

        if horizon < 0:
            raise ValueError("horizon must be >= 0.")            
        self.horizon = horizon
        self.drop_boundary = drop_boundary + horizon

        policies = {"merge", "drop", "pad"}
        if tail_policy not in ("merge", "drop", "pad"):
            print(f"{tail_policy} not in {policies}, fall back to drop tail.")
            tail_policy = "drop"
        self.tail_policy = tail_policy

        all_subjects = self.metadata[self.subject_col_meta].astype(str).tolist()
        if all_subjects is None:
            raise KeyError(f"No subjects found from {metadata_csv}.")
        else:
            allowed_set = set(map(str, allowed_subjects))
            all_subjects = [sid for sid in all_subjects if sid in allowed_set]
        self.subject_ids = all_subjects

        self.meta_by_subject = (
            self.metadata.assign(**{self.subject_col_meta: self.metadata[self.subject_col_meta].astype(str)})
            .set_index(self.subject_col_meta)
            .to_dict(orient="index")
        )

        # cache for loaded df
        self._cache: dict[str, pd.DataFrame] = {}

        if self.preload:
            for sid in self.subject_ids:
                self._cache[sid] = self._load_subject_df(sid)
        
        # build block index
        self.all_blocks: list[BlockSample] = self._build_index()
        
    # -- subject loading --
    def _subject_path(self, subject_id: str) -> Path:
        return self.signals_dir / self.signals_file_pattern.format(subject_id=subject_id)
    
    def _load_subject_df(self, subject_id: str) -> pd.DataFrame:
        path = self._subject_path(subject_id)
        if not path.exists():
            raise FileNotFoundError(f"Missing parquet for subject {subject_id}: {path}")
        df = pd.read_parquet(path).copy()

        # check columns
        required = set(self.feature_cols + self.label_cols)
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{path} file missing columns: {sorted(missing)}")
        
        # assuming rows are already sorted as time ordered. 
        return df
    
    # -- block indexing --
    def _get_subject_df(self, subject_id: str) -> pd.DataFrame:
        if subject_id not in self._cache:
            self._cache[subject_id] = self._load_subject_df(subject_id)
        return self._cache[subject_id]
    
    def _build_index(self) -> list[BlockSample]:
        all_blocks: list[BlockSample] = []
        needed_block_len = 2 * self.drop_boundary
        if self.block_size < needed_block_len:
            raise ValueError(
                f"block_size={self.block_size} is too small for drop_boundary={needed_block_len}"
            )
        # TODO: implement different tail strategies. Everything leads to "drop" tail for now.
        for sid in self.subject_ids:
            df = self._get_subject_df(sid)
            n = len(df)
            n_blocks = n // self.block_size
            mod_blocks = n % self.block_size
            for b in range(n_blocks):
                start = b * self.block_size
                end = start + self.block_size
                all_blocks.append(
                    BlockSample(
                        subject_id=sid,
                        block_id=b,
                        start_idx=start,
                        end_idx=end,
                    )
                )
        return all_blocks
    
    # -- purge boundaries --
    def __len__(self) -> int:
        return len(self.all_blocks)
    def __getitem__(self, idx: int):
        """get X, y, meta from single block"""
        block = self.all_blocks[idx]
        df = self._get_subject_df(block.subject_id)
        block_df = df.iloc[block.start_idx : block.end_idx].reset_index(drop=True)

        # purge boundaies
        if self.drop_boundary > 0:
            block_df = block_df.iloc[self.drop_boundary : len(block_df)].reset_index(drop=True)
        X = block_df[self.feature_cols].to_numpy(dtype=np.float32)
        y = block_df[self.label_cols].to_numpy()

        # apply horizon shift
        if self.horizon > 0:
            if len(block_df) <= self.horizon:
                raise ValueError(f"Block sid={block.subject_id}, block_id={block.block_id} is too short for horizon={self.horizon}.")
            X = X[:-self.horizon]
            y = y[self.horizon:]
        
        # apply meta data
        subject_meta = self.meta_by_subject.get(block.subject_id, {})
        # TODO: customize meata features to include
        meta_feature_cols = ['age', 'bmi', 'oahi', 'ahi', 'mean_sao2', 'arousal_index', 'gender_m', 'med_anxiety', 'med_arrhythmia', 'med_asthma', 'med_body_pain', 'med_cad', 'med_depression', 'med_diabetes', 'med_dyspnea', 'med_gerd', 'med_hypertension', 'med_migraine', 'med_osa']
        meta_values = np.array(
            [subject_meta.get(col, np.nan) for col in meta_feature_cols],
            dtype=np.float32,
        )
        meta_matrix = np.tile(meta_values, (X.shape[0],1))
        X = np.hstack([X,meta_matrix])

        meta = {
            "subject_id": block.subject_id,
            "block_id" : block.block_id,
            "start_idx" : block.start_idx,
            "end_idx" : block.end_idx,
            "n_rows_raw": block.end_idx - block.start_idx,
            "n_rows_final": len(y),
            "subject_meta": self.meta_by_subject.get(block.subject_id, {}),
        }

        return X, y, meta

In [4]:
# --- Helper Functions ---

# -- leave n subject out --
from collections import defaultdict


def make_leave_n_out_split(
        subject_ids: Sequence[str],
        n_test: int,
        seed: int = 42,
) -> tuple[list[str], list[str]]:
    subject_ids = list(map(str,subject_ids))
    if not (1 <= n_test < len(subject_ids)):
        raise ValueError("n_test must be between 1 and len(subject_ids)-1.")
    rng = random.Random(seed)
    shuffled = subject_ids[:]
    rng.shuffle(shuffled)
    test_subjects = shuffled[:n_test]
    train_subjects = shuffled[n_test:]
    return train_subjects, test_subjects

# -- split blocks into train,val,test within subject --
def split_within_subject(
        dataset: SleepDataset,
        ratio: Sequence[float]=(0.6,0.2,0.2),
        seed: int = 42,
) -> list[int]:
    
    # Group dataset
    subject_blocks = defaultdict(list)
    for idx, block in enumerate(dataset.all_blocks):
        subject_blocks[block.subject_id].append(idx)
    
    train_ids = []
    val_ids = []
    test_ids = []

    # shuffle and collect block indices
    for subject_id, indices in subject_blocks.items():
        # sort indices by block_id
        indices = sorted(indices, key=lambda i: dataset.all_blocks[i].block_id)
        subject_seed = seed + int(subject_id[1:])*2
        rng = random.Random(subject_seed)
        indices = indices[:]
        rng.shuffle(indices)

        # split indices
        n = len(indices)
        if n < 3:
            print(f"Subject {subject_id} has only {n} blocks, skip train-val-test split.")
            return None

        n_train = round(n*ratio[0])
        n_val = round(n*ratio[1])
        n_test = n-n_train-n_val

        if min(n_train, n_val, n_test) == 0:
            print(
                f"Subject {subject_id} has too few blocks ({n}) for ratios={ratio}. "
                f"Got split sizes train={n_train}, val={n_val}, test={n_test}."
            )

        train_ids.extend(indices[:n_train])
        val_ids.extend(indices[n_train:n_train+n_val])
        test_ids.extend(indices[n_train + n_val:])

    return train_ids, val_ids, test_ids
    
    

# -- make subset --
class DatasetSubset:
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = list(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        return self.dataset[self.indices[i]]

# -- flatten dataset --
def flatten_dataset(dataset: SleepDataset | DatasetSubset):
    X_list = []
    y_list = []
    rows = []

    for i in range(len(dataset)):
        X_block, y_block, meta = dataset[i]
        X_list.append(X_block)
        y_list.append(y_block)
        
        # multiply metat data by rows of block
        rows.extend(
            {
                "subject_id": meta["subject_id"],
                "block_id": meta["block_id"],
                "row_in_block": row_idx,
            }
            for row_idx in range(len(y_block))  
        )
    X = np.vstack(X_list)
    y = np.vstack(y_list)
    info = pd.DataFrame(rows)
    return X, y, info

# -- encode sleep_stage col --
STAGE_MAP = {
    "W": 0,
    "N1": 1,
    "N2": 2,
    "N3": 3,
    "R": 4,
}
def encode_labels(y, stage_map=STAGE_MAP):
    y = np.asarray(y).flatten()
    return np.array([stage_map[s] for s in y], dtype=np.int64)

# Check variables and load dataset

In [7]:
# define columns
df = pd.read_parquet("../data/processed/data_100Hz/S002.parquet")
print(list(df.columns))
print(df[['epoch_id','sleep_stage']].head())

['epoch_id', 'epoch_start', 'epoch_end', 'eeg_c4_mean', 'eeg_c4_std', 'eeg_c4_min', 'eeg_c4_max', 'eeg_c4_range', 'eeg_c4_slope', 'eeg_f4_mean', 'eeg_f4_std', 'eeg_f4_min', 'eeg_f4_max', 'eeg_f4_range', 'eeg_f4_slope', 'eeg_o2_mean', 'eeg_o2_std', 'eeg_o2_min', 'eeg_o2_max', 'eeg_o2_range', 'eeg_o2_slope', 'eeg_fp1_mean', 'eeg_fp1_std', 'eeg_fp1_min', 'eeg_fp1_max', 'eeg_fp1_range', 'eeg_fp1_slope', 'eeg_t3_mean', 'eeg_t3_std', 'eeg_t3_min', 'eeg_t3_max', 'eeg_t3_range', 'eeg_t3_slope', 'eeg_cz_mean', 'eeg_cz_std', 'eeg_cz_min', 'eeg_cz_max', 'eeg_cz_range', 'eeg_cz_slope', 'eog_e1_mean', 'eog_e1_std', 'eog_e1_min', 'eog_e1_max', 'eog_e1_range', 'eog_e1_slope', 'eog_e2_mean', 'eog_e2_std', 'eog_e2_min', 'eog_e2_max', 'eog_e2_range', 'eog_e2_slope', 'emg_chin_mean', 'emg_chin_std', 'emg_chin_min', 'emg_chin_max', 'emg_chin_range', 'emg_chin_slope', 'emg_lat_mean', 'emg_lat_std', 'emg_lat_min', 'emg_lat_max', 'emg_lat_range', 'emg_lat_slope', 'emg_rat_mean', 'emg_rat_std', 'emg_rat_min',

In [9]:
from pathlib import Path
import re

signals_dir = Path('../data/processed/data_100Hz/')
pattern = re.compile(r'^(S\d{3,4})')  # matches S followed by 3-4 digits

all_subject_ids = []
for p in signals_dir.iterdir():
    if not p.is_file():
        continue
    m = pattern.match(p.stem)
    if m:
        all_subject_ids.append(m.group(1))

# unique + numeric sort by the number after 'S'
all_subject_ids = sorted(set(all_subject_ids), key=lambda s: int(s[1:]))

print(all_subject_ids)

['S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020', 'S021', 'S022', 'S023', 'S024', 'S025', 'S026', 'S027', 'S028', 'S029', 'S030', 'S031', 'S032', 'S033', 'S034', 'S035', 'S036', 'S037', 'S038', 'S039', 'S040', 'S042', 'S043', 'S044', 'S045', 'S046', 'S047', 'S048', 'S049', 'S050', 'S051', 'S052', 'S053', 'S054', 'S055', 'S056', 'S057', 'S058', 'S059', 'S061', 'S062', 'S063', 'S064', 'S065', 'S066', 'S067', 'S068', 'S069', 'S070', 'S071', 'S072', 'S073', 'S074', 'S075', 'S076', 'S077', 'S078', 'S079', 'S080', 'S081', 'S082', 'S083', 'S084', 'S085', 'S086', 'S087', 'S088', 'S089', 'S090', 'S091', 'S092', 'S093', 'S094', 'S095', 'S096', 'S097', 'S098', 'S099', 'S100', 'S101', 'S102', 'S103']


In [12]:
df = pd.read_csv('../data/processed/participant_info_preprocessed.csv')
print(list(df.columns))

['sid', 'age', 'bmi', 'oahi', 'ahi', 'mean_sao2', 'arousal_index', 'gender_m', 'sd_bruxism', 'sd_dyspnea', 'sd_eds', 'sd_fatigue', 'sd_hypersomnia', 'sd_insomnia', 'sd_mci_and_sleep_apnea', 'sd_migraine', 'sd_osa', 'sd_osa_snoring', 'sd_rbd', 'sd_rls', 'sd_snoring', 'med_anxiety', 'med_arrhythmia', 'med_asthma', 'med_body_pain', 'med_cad', 'med_depression', 'med_diabetes', 'med_dyspnea', 'med_gerd', 'med_hypertension', 'med_migraine', 'med_osa']


In [15]:
# -- test on small batch or entire dataset --
SMALL_BATCH_TEST = True # toggle on/off

feat_cols = ['eeg_c4_mean', 'eeg_c4_std', 'eeg_c4_min', 'eeg_c4_max', 'eeg_c4_range', 'eeg_c4_slope', 'eeg_f4_mean', 'eeg_f4_std', 'eeg_f4_min', 'eeg_f4_max', 'eeg_f4_range', 'eeg_f4_slope', 'eeg_o2_mean', 'eeg_o2_std', 'eeg_o2_min', 'eeg_o2_max', 'eeg_o2_range', 'eeg_o2_slope', 'eeg_fp1_mean', 'eeg_fp1_std', 'eeg_fp1_min', 'eeg_fp1_max', 'eeg_fp1_range', 'eeg_fp1_slope', 'eeg_t3_mean', 'eeg_t3_std', 'eeg_t3_min', 'eeg_t3_max', 'eeg_t3_range', 'eeg_t3_slope', 'eeg_cz_mean', 'eeg_cz_std', 'eeg_cz_min', 'eeg_cz_max', 'eeg_cz_range', 'eeg_cz_slope', 'eog_e1_mean', 'eog_e1_std', 'eog_e1_min', 'eog_e1_max', 'eog_e1_range', 'eog_e1_slope', 'eog_e2_mean', 'eog_e2_std', 'eog_e2_min', 'eog_e2_max', 'eog_e2_range', 'eog_e2_slope', 'emg_chin_mean', 'emg_chin_std', 'emg_chin_min', 'emg_chin_max', 'emg_chin_range', 'emg_chin_slope', 'emg_lat_mean', 'emg_lat_std', 'emg_lat_min', 'emg_lat_max', 'emg_lat_range', 'emg_lat_slope', 'emg_rat_mean', 'emg_rat_std', 'emg_rat_min', 'emg_rat_max', 'emg_rat_range', 'emg_rat_slope', 'resp_ptaf_mean', 'resp_ptaf_std', 'resp_ptaf_min', 'resp_ptaf_max', 'resp_ptaf_range', 'resp_ptaf_slope', 'resp_flow_mean', 'resp_flow_std', 'resp_flow_min', 'resp_flow_max', 'resp_flow_range', 'resp_flow_slope', 'resp_thorax_mean', 'resp_thorax_std', 'resp_thorax_min', 'resp_thorax_max', 'resp_thorax_range', 'resp_thorax_slope', 'resp_abdomen_mean', 'resp_abdomen_std', 'resp_abdomen_min', 'resp_abdomen_max', 'resp_abdomen_range', 'resp_abdomen_slope', 'bvp_mean', 'bvp_std', 'bvp_min', 'bvp_max', 'bvp_range', 'bvp_slope', 'ibi_mean', 'ibi_std', 'ibi_min', 'ibi_max', 'ibi_range', 'ibi_slope', 'eda_mean', 'eda_std', 'eda_min', 'eda_max', 'eda_range', 'eda_slope', 'temp_mean', 'temp_std', 'temp_min', 'temp_max', 'temp_range', 'temp_slope', 'hr_mean', 'hr_std', 'hr_min', 'hr_max', 'hr_range', 'hr_slope', 'snore_mean', 'snore_std', 'snore_min', 'snore_max', 'snore_range', 'snore_slope', 'sao2_mean', 'sao2_std', 'sao2_min', 'sao2_max', 'sao2_range', 'sao2_slope', 'eeg_c4_bp_delta', 'eeg_c4_bp_theta', 'eeg_c4_bp_alpha', 'eeg_c4_bp_sigma', 'eeg_c4_bp_beta', 'eeg_c4_bp_gamma', 'eeg_c4_bp_high_gamma', 'eeg_f4_bp_delta', 'eeg_f4_bp_theta', 'eeg_f4_bp_alpha', 'eeg_f4_bp_sigma', 'eeg_f4_bp_beta', 'eeg_f4_bp_gamma', 'eeg_f4_bp_high_gamma', 'eeg_o2_bp_delta', 'eeg_o2_bp_theta', 'eeg_o2_bp_alpha', 'eeg_o2_bp_sigma', 'eeg_o2_bp_beta', 'eeg_o2_bp_gamma', 'eeg_o2_bp_high_gamma', 'eeg_fp1_bp_delta', 'eeg_fp1_bp_theta', 'eeg_fp1_bp_alpha', 'eeg_fp1_bp_sigma', 'eeg_fp1_bp_beta', 'eeg_fp1_bp_gamma', 'eeg_fp1_bp_high_gamma', 'eeg_t3_bp_delta', 'eeg_t3_bp_theta', 'eeg_t3_bp_alpha', 'eeg_t3_bp_sigma', 'eeg_t3_bp_beta', 'eeg_t3_bp_gamma', 'eeg_t3_bp_high_gamma', 'eeg_cz_bp_delta', 'eeg_cz_bp_theta', 'eeg_cz_bp_alpha', 'eeg_cz_bp_sigma', 'eeg_cz_bp_beta', 'eeg_cz_bp_gamma', 'eeg_cz_bp_high_gamma', 'eog_e1_bp_slow', 'eog_e1_bp_delta', 'eog_e1_bp_theta', 'eog_e1_bp_high', 'eog_e2_bp_slow', 'eog_e2_bp_delta', 'eog_e2_bp_theta', 'eog_e2_bp_high', 'emg_chin_bp_low', 'emg_chin_bp_medium', 'emg_chin_bp_high', 'emg_lat_bp_low', 'emg_lat_bp_medium', 'emg_lat_bp_high', 'emg_rat_bp_low', 'emg_rat_bp_medium', 'emg_rat_bp_high', 'ibi_sdnn', 'ibi_rmssd', 'ibi_pnn50', 'eda_scr_count', 'eda_scr_mean_amp', 'hr_bvp_corr', 'acc_eda_corr']
target_col = ['sleep_stage']
signals_dir = '../data/processed/data_100Hz/'
meta_csv = '../data/processed/participant_info_preprocessed.csv'

subjects = ['S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020', 'S021', 'S022', 'S023', 'S024', 'S025', 'S026', 'S027', 'S028', 'S029', 'S030', 'S031', 'S032', 'S033', 'S034', 'S035', 'S036', 'S037', 'S038', 'S039', 'S040', 'S042', 'S043', 'S044', 'S045', 'S046', 'S047', 'S048', 'S049', 'S050', 'S051', 'S052', 'S053', 'S054', 'S055', 'S056', 'S057', 'S058', 'S059', 'S061', 'S062', 'S063', 'S064', 'S065', 'S066', 'S067', 'S068', 'S069', 'S070', 'S071', 'S072', 'S073', 'S074', 'S075', 'S076', 'S077', 'S078', 'S079', 'S080', 'S081', 'S082', 'S083', 'S084', 'S085', 'S086', 'S087', 'S088', 'S089', 'S090', 'S091', 'S092', 'S093', 'S094', 'S095', 'S096', 'S097', 'S098', 'S099', 'S100', 'S101', 'S102', 'S103']
if SMALL_BATCH_TEST:
    subjects = subjects[:6]

train_subs, test_subs = make_leave_n_out_split(subjects,n_test=2)
train_pool_ds = SleepDataset(
    signals_dir=signals_dir,
    metadata_csv=meta_csv,
    feature_cols=feat_cols,
    label_cols=target_col,
    allowed_subjects=subjects,
)
train_idx, val_idx, dev_test_idx = split_within_subject(
    train_pool_ds,
    ratio=(0.6, 0.2, 0.2),
)
train_subset = DatasetSubset(train_pool_ds,train_idx)
val_subset = DatasetSubset(train_pool_ds,val_idx)
dev_test_subset = DatasetSubset(train_pool_ds,dev_test_idx)

X_train, y_train, info_train = flatten_dataset(train_subset)
X_val, y_val, info_val = flatten_dataset(val_subset)
X_dev_test, y_dev_test, info_dev_test = flatten_dataset(dev_test_subset)

# for 'sleep_stage' detection only
y_train = encode_labels(y_train)
y_val = encode_labels(y_val)
y_dev_test = encode_labels(y_dev_test)


# Model train and eval

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

logreg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
                max_iter=2000,
                solver="sag",
                class_weight="balanced",
                random_state=42
            ))
])

logreg.fit(X_train, y_train)
val_pred = logreg.predict(X_val)

print("Val macro F1:", f1_score(y_val, val_pred, average="macro"))
print(classification_report(y_val, val_pred, digits=4))

Val macro F1: 0.6059673759656101
              precision    recall  f1-score   support

           0     0.6967    0.6726    0.6844      1298
           1     0.1411    0.3211    0.1960       355
           2     0.8791    0.6443    0.7436      2901
           3     0.5686    0.9395    0.7085       397
           4     0.6523    0.7490    0.6973       729

    accuracy                         0.6646      5680
   macro avg     0.5876    0.6653    0.6060      5680
weighted avg     0.7405    0.6646    0.6875      5680



/home/aiqi-cheng/miniconda3/envs/sleep_project/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [17]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    random_state=42,
)

xgb.fit(X_train, y_train)

val_pred = xgb.predict(X_val)

print("Val macro F1:", f1_score(y_val, val_pred, average="macro"))
print(classification_report(y_val, val_pred, digits=4))

Val macro F1: 0.7104766111237808
              precision    recall  f1-score   support

           0     0.7818    0.8891    0.8320      1298
           1     0.5667    0.2394    0.3366       355
           2     0.8255    0.9228    0.8714      2901
           3     0.9321    0.6574    0.7710       397
           4     0.8795    0.6406    0.7413       729

    accuracy                         0.8176      5680
   macro avg     0.7971    0.6699    0.7105      5680
weighted avg     0.8137    0.8176    0.8053      5680

